In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import matplotlib.pyplot as plt


In [2]:
from sklearn.datasets import fetch_openml
elec_data = fetch_openml(name='electricity', version=1)
elec_df = elec_data.frame
print('Data shape:', elec_data.data.shape)
print('Feature names:', elec_data.feature_names)
print('Target names:', elec_data.target_names)



Data shape: (45312, 8)
Feature names: ['date', 'day', 'period', 'nswprice', 'nswdemand', 'vicprice', 'vicdemand', 'transfer']
Target names: ['class']


In [3]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    elec_df, test_size=0.25, random_state=20
)
print('Training samples:', len(train_data))
print('Testing samples:', len(test_data))


Training samples: 33984
Testing samples: 11328


In [4]:
from sklearn.preprocessing import LabelEncoder

X_train = train_data.drop('class', axis=1)
y_train = train_data['class']

label_enc = LabelEncoder()
y_train_prepared = label_enc.fit_transform(y_train)
print(label_enc.classes_)


['DOWN' 'UP']


In [5]:
from sklearn.metrics import accuracy_score
def accuracy(model, data, labels):
  predictions = model.predict(data)
  acc = accuracy_score(labels, predictions)
  return acc


In [6]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [7]:
X_train_num = X_train.copy()
X_train_num['day'] = X_train_num['day'].astype(int)

models = [
    ('lr', LogisticRegression(max_iter=1000)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('svc', SVC(probability=True, random_state=42)),
]

voting_hard = VotingClassifier(models, voting='hard')
voting_soft = VotingClassifier(models, voting='soft')

voting_hard.fit(X_train_num, y_train_prepared)
voting_soft.fit(X_train_num, y_train_prepared)

print(accuracy(voting_hard, X_train_num, y_train_prepared))
print(accuracy(voting_soft, X_train_num, y_train_prepared))


0.8177083333333334
0.9141949152542372
